In [3]:
pip show openai qdrant-client python-dotenv


Name: openai
Version: 3.42.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License-Expression: Apache-2.0
Location: c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages
Requires: anyio, httpx2, jiter, pydantic, sniffio, typing-extensions
Required-by: 
---
Name: qdrant-client
Version: 1.20.0
Summary: Client library for the Qdrant vector search engine
Home-page: https://github.com/qdrant/qdrant-client
Author: Andrey Vasnetsov
Author-email: andrey@qdrant.tech
License: Apache-2.0
Location: c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages
Requires: grpcio, httpx, numpy, portalocker, protobuf, pydantic, urllib3
Required-by: 
---
Name: python-dotenv
Version: 1.2.4
Summary: Read key-value pairs from a .env file and set them as environment variables
Home-page: 
Author: 
Author-email: Saurabh Kumar <me+github@saurabh-kumar.com>
License: BSD-3-C

In [ ]:
qdrant_api_key = "qdrant_api_key"
qdrant_url = "https://b27f44e8-a944-433f-a868-c0a7d1eb0702.us-west-1-0.aws.cloud.qdrant.io"
euri_api_key = "euri-key"

In [2]:
pip show qdrant_client

Name: qdrant-client
Version: 1.20.0
Summary: Client library for the Qdrant vector search engine
Home-page: https://github.com/qdrant/qdrant-client
Author: Andrey Vasnetsov
Author-email: andrey@qdrant.tech
License: Apache-2.0
Location: c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages
Requires: grpcio, httpx, numpy, portalocker, protobuf, pydantic, urllib3
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import re
import base64
import mimetypes
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient,models
CORPUS_PATH = Path(r"E:\euron-super30\supabase_db_kit\data\corpus")
euri = OpenAI(
    api_key=euri_api_key,
    base_url="https://api.euron.one/api/v1/euri"
)

In [9]:
from qdrant_client.models import Distance, VectorParams, PointStruct
client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key, timeout=60)

COLLECTION = "embedding_collection"
if client.collection_exists(collection_name=COLLECTION):
    client.delete_collection(collection_name=COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
)
MODEL = "gemini-embedding-2"

In [10]:
def clean_text(text: str) -> str:
    """Cleans raw text by stripping extra whitespace and boilerplate headers."""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'Page \d+ of \d+', '', text)
    return text.strip()

def chunk_text(text: str, chunk_size: int = 800, overlap: int = 150) -> list[str]:
    """Splits text into sliding window chunks."""
    cleaned = clean_text(text)
    chunks = []
    start = 0
    while start < len(cleaned):
        end = start + chunk_size
        chunks.append(cleaned[start:end])
        start += chunk_size - overlap
    return chunks

In [13]:
import json
def load_corpus_from_directory(directory_path: Path, target_count: int = 200) -> list[dict]:
    """Loads at least target_count records from JSONL files in local corpus subfolders[cite: 2, 3]."""
    documents = []
    if not directory_path.exists():
        print(f"Directory not found: {directory_path}")
        return documents

    # Recursively look for .jsonl files
    for file_path in directory_path.glob("**/*.jsonl"):
        try:
            path_parts = file_path.parts
            doc_type = "general_corpus"
            if "rba_guides" in path_parts:
                doc_type = "rba_guides"
            elif "investigator_notes" in path_parts:
                doc_type = "investigator_notes"
            elif "playbooks" in path_parts:
                doc_type = "playbooks"
            elif "chargeback_narratives" in path_parts:
                doc_type = "chargeback_narratives"
            elif "guidelines" in path_parts:
                doc_type = "guidelines"
            elif "regulatory_exercises" in path_parts:
                doc_type = "regulatory_exercises"

            with open(file_path, "r", encoding="utf-8") as f:
                for line_idx, line in enumerate(f):
                    if not line.strip():
                        continue
                    record = json.loads(line)
                    
                    # Extract text content from common JSON keys
                    content = record.get("text") or record.get("content") or record.get("narrative") or json.dumps(record)
                    
                    documents.append({
                        "doc_id": f"{file_path.stem}_line_{line_idx + 1}",
                        "doc_type": doc_type,
                        "content": content
                    })
                    
                    # Stop once we have reached or exceeded the target count
                    if len(documents) >= target_count:
                        return documents
        except Exception as e:
            print(f"Error reading {file_path.name}: {e}")
                
    return documents

# --- Step 1: Load at least 200 Records ---
print(f"Loading corpus from {CORPUS_PATH}...")
raw_documents = load_corpus_from_directory(CORPUS_PATH, target_count=200)
print(f"Successfully loaded {len(raw_documents)} raw source documents.")

Loading corpus from E:\euron-super30\supabase_db_kit\data\corpus...
Successfully loaded 200 raw source documents.


In [14]:
# --- Step 2: Chunk Documents ---
processed_chunks = []
for doc in raw_documents:
    chunks = chunk_text(doc["content"])
    for idx, chunk in enumerate(chunks):
        processed_chunks.append({
            "source_document": doc["doc_id"],
            "document_type": doc["doc_type"],
            "chunk_number": idx + 1,
            "text": chunk
        })

print(f"Generated {len(processed_chunks)} total chunks from the records.")

Generated 622 total chunks from the records.


In [31]:
pip install google-genai

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached charset_normalizer-3.5.2-cp314-cp314-win_amd64.whl.metadata (47 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------- ----------------------- 0.5/1.3 MB 2.6 MB/s eta 0:00:01
   ------------------------- -------------- 0.8/1.3 MB 2.5 MB/s eta 0:00:01
   --------------------------------- ------ 1.0/1.3 MB 2.1 MB/s eta 0:00:01
   --------------------------------- ------ 1.0/1.3 MB 2.1 MB/s eta 0:00:01
   --------------------------------- ------ 1.0/1.3 MB 2.1 MB/s eta 0:00:01
   --------------------------------- ------ 1.0/1.3 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 743.3 kB/s  0:00:01
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.2-cp314-cp314-win_amd64.whl (210 kB)
Using cached pyasn1_modules-0.4.2-py3-none-any.whl (1


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# --- Step 4: Embed & Upsert in Batches ---
from google import genai
from google.genai import types
from qdrant_client.models import Distance, VectorParams, PointStruct
ai_client = genai.Client(api_key="apikey")
import time
from google.genai.errors import APIError

batch_size = 10 

for i in range(0, len(processed_chunks), batch_size):
    batch = processed_chunks[i:i + batch_size]
    points_to_upsert = []
    
    for item in batch:
        success = False
        retries = 3
        wait_time = 2
        
        while retries > 0 and not success:
            try:
                response = ai_client.models.embed_content(
                    model="gemini-embedding-2",
                    contents=item["text"],
                    config=types.EmbedContentConfig(
                        task_type="RETRIEVAL_DOCUMENT",
                        output_dimensionality=3072
                    )
                )
                vector = response.embeddings[0].values
                success = True
            except APIError as e:
                if e.code == 503:
                    print(f"Gemini 503 Service Unavailable. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                    retries -= 1
                    wait_time *= 2
                else:
                    raise e
            except Exception as e:
                print(f"Embedding error: {e}. Retrying in {wait_time}s...")
                time.sleep(wait_time)
                retries -= 1
                wait_time *= 2

        if not success:
            raise Exception("Failed to fetch embedding after multiple retries.")

        points_to_upsert.append(
            PointStruct(
                id=str(os.urandom(16).hex()),
                vector=vector,
                payload=item
            )
        )
    
    # Upsert smaller batch with a safety catch for Qdrant write timeouts
    try:
        client.upsert(collection_name=COLLECTION, points=points_to_upsert, wait=True)
        print(f"Successfully upserted batch {i // batch_size + 1} ({len(points_to_upsert)} points).")
    except Exception as e:
        print(f"Qdrant write timeout/error on batch: {e}. Retrying batch once...")
        time.sleep(5)
        client.upsert(collection_name=COLLECTION, points=points_to_upsert, wait=True)
        print(f"Successfully upserted batch {i // batch_size + 1} on retry.")

    time.sleep(1)  # Brief pause between batches

print("Ingestion completed successfully without timeouts!")

Successfully upserted batch 1 (10 points).
Successfully upserted batch 2 (10 points).
Successfully upserted batch 3 (10 points).
Successfully upserted batch 4 (10 points).


KeyboardInterrupt: 

In [17]:
from qdrant_client import models

# Create payload index for document_type
client.create_payload_index(
    collection_name=COLLECTION,
    field_name="document_type",
    field_schema=models.PayloadSchemaType.KEYWORD,
)
print("Payload index created successfully for 'document_type'!")

Payload index created successfully for 'document_type'!


In [20]:
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from google import genai
from google.genai import types
import os
import httpx


def get_query_embedding(text: str) -> list[float]:
    """Generates a query embedding using Gemini Embedding 2 with RETRIEVAL_QUERY task type."""
    response = ai_client.models.embed_content(
        model="gemini-embedding-2",
        contents=text,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=3072
        )
    )
    return response.embeddings[0].values

def search_qdrant(query_text: str, top_k: int = 5, query_filter: Filter = None):
    """Embeds query and executes vector similarity search on Qdrant."""
    query_vector = get_query_embedding(query_text)
    
    search_result = client.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        query_filter=query_filter,
        limit=top_k,
        with_payload=True
    ).points
    
    return search_result

# --- 10 Natural-Language Queries ---
queries = [
    #"what should an investigator check when a new device makes a large payment",
    #"what is the cooling period for newly bound UPI accounts according to RBI",
    # "how to handle chargeback claims for fraudulent card transactions",
    # "what risk flags indicate automated script or bot activity in online payments",
    # "what are the NPCI mandates for recurring UPI auto-debit notification",
    # "steps to trace mule bank accounts involved in cyber fraud",
     "what documentation is needed to escalate a high-value payment dispute",
    # "how does device fingerprinting assist in identifying account takeover",
    # "what are the turnaround time SLAs for resolving chargeback disputes",
    # "what constitutes unauthorized electronic banking transaction liability"
]

print("=== Running 10 Natural-Language Searches ===")
for i, query in enumerate(queries, start=1):
    print(f"\nQuery {i}: '{query}'")
    results = search_qdrant(query, top_k=5)
    for rank, point in enumerate(results, start=1):
        print(f"  Top {rank} [Score: {point.score:.4f}] | Source: {point.payload['source_document']} | Type: {point.payload['document_type']}")
        print(f"    Excerpt: {point.payload['text'][:120]}...")

# --- Payload Filtered Search Example ---
print("\n=== Running Filtered Search (document_type == 'investigator_notes') ===")
filtered_query = "what should an investigator check when a new device makes a large payment"

investigator_filter = Filter(
    must=[
        FieldCondition(
            key="document_type",
            match=MatchValue(value="investigator_notes")
        )
    ]
)

filtered_results = search_qdrant(filtered_query, top_k=5, query_filter=investigator_filter)

for rank, point in enumerate(filtered_results, start=1):
    print(f"  Top {rank} [Score: {point.score:.4f}] | Source: {point.payload['source_document']} | Type: {point.payload['document_type']}")
    print(f"    Excerpt: {point.payload['text'][:120]}...")

=== Running 10 Natural-Language Searches ===

Query 1: 'what documentation is needed to escalate a high-value payment dispute'


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-2, description: Embed content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 48m53.16997739s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Embed content requests per day limit exceeded on Free Tier. See https://ai.